In [ ]:
import torch
import torch.nn as nn
from typing import Optional
import matplotlib.pyplot as plt
import torch.optim as optim
import math
from torch.utils.data import DataLoader
import os
from google.colab import drive
import numpy as np
import random
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import time
import json
from scipy.spatial import distance
import numpy as np
import pandas as pd
from scipy.interpolate import splprep, splev

In [ ]:
# device = torch.device("cuda")
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=32, output_dim=2, num_layers=3):
        super(Seq2Seq, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.output_dim = output_dim
        self.input_dim = input_dim
        # Encoder
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)

        # Decoder
        self.decoder = nn.LSTM(hidden_dim + output_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

        # Store hidden and cell states
        self.hidden = None
        self.cell = None

    def forward(self, x, target_seq_len = 15):
        batch_size, seq_len, _ = x.shape


        self.hidden, self.cell = None,None #This line can be changed to make the model persists state between forward passes.


        # Encode
        _, (hidden, cell) = self.encoder(x, (self.hidden, self.cell) if self.hidden is not None else None)



        # Prepare decoder input (first input as zeros)
        decoder_input = torch.zeros(batch_size, 1, self.hidden_dim + self.output_dim, device=x.device)
        outputs = []

        for _ in range(target_seq_len):
            decoder_output, (hidden, cell) = self.decoder(decoder_input, (hidden, cell))
            output = self.fc(decoder_output[:, -1, :])  # Get last time step output
            outputs.append(output.unsqueeze(1))

            # Update decoder input with the last output
            decoder_input = torch.cat((decoder_output, output.unsqueeze(1)), dim=-1)

        return torch.cat(outputs, dim=1)

In [19]:
class SeqDataset(Dataset):
    def __init__(self,src_path, tgt_path,src2_path, input_size=8, target_size=25):
        self.training_size = len(os.listdir(src_path))
        self.training_size2 = len(os.listdir(src2_path))
        self.src_files = [f'{src_path}/src{n}.pt' for n in range(self.training_size)]
        self.tgt_files = [f'{tgt_path}/tgt{n}.pt' for n in range(self.training_size)]
        self.csv_files = [f'{src2_path}/random_track{n}.csv' for n in range(self.training_size2)]
        self.input_size = input_size
        self.target_size = target_size

    def __len__(self):
        return len(self.src_files)

    def __getitem__(self, index):
        src = torch.load(self.src_files[index])  # Shape: [seq_len, feat_dim]
        tgt = torch.load(self.tgt_files[index])  # Shape: [seq_len, feat_dim]

        df = pd.read_csv(self.csv_files[index%self.training_size2], header=None)
        df.drop(df.columns[[3,4,5,6]],axis=1,inplace=True)
        df.columns = ['color', 'x', 'y']

        left2 = df[df['color'] == 'blue'][['x', 'y']].to_numpy() 
        left2 = np.concatenate([left2,np.ones((left2.shape[0], 1)), np.zeros((left2.shape[0], 1))], axis=1)
        right2 = df[df['color'] == 'yellow'][['x', 'y']].to_numpy()
        right2 = np.concatenate([right2, np.zeros((right2.shape[0], 1)), np.ones((right2.shape[0], 1))], axis=1)

        tgt2 = generate_mid_track(left2, right2)
        tgt2 = torch.from_numpy(tgt2)


        

        # Create masks
        left_mask = src[:, 2] == 1
        right_mask = src[:, 4] == 1

        # Apply masks and padding
        left = pad_and_mask(src, left_mask)
        right = pad_and_mask(src, right_mask)

        # Interleave
        left_selected = left[:, [0, 1,2, 4]]  # shape [N, 3]
        right_selected = right[:, [0,1,2, 4]]

        left2 = torch.from_numpy(left2)
        right2 = torch.from_numpy(right2)



        seq = fast_interleave(left_selected, right_selected)
        seq2 = fast_interleave(left2, right2)
        #seq = interleave(left, right)  # Shape: [new_seq_len, feat_dim]

        # Create sliding window samples
        samples = []
        if random.random() > 0.5:
            seq, tgt = reverse_track(seq, tgt)
            seq2, tgt2 = reverse_track(seq2, tgt2)
            
        k=0
        while k<tgt.size(0):
            target_chunk = tgt[k:k+self.target_size, :]
            start = target_chunk[0]
            dx = target_chunk[1][0] - target_chunk[0][0]
            dy = target_chunk[1][1] - target_chunk[0][1]
            theta = math.atan2(dy, dx)
            input_chunk = seq[is_inside_semicircle(start, theta, 8, seq)][:self.input_size,:]
            input_chunk, target_chunk = normalize_data(input_chunk, target_chunk, start, theta)
            input_chunk = add_noisy_data(input_chunk)
            samples.append((input_chunk[torch.randperm(input_chunk.size(0))] , target_chunk))
            k+=5

        k=0
        while k<tgt2.size(0):
            target_chunk = tgt2[k:k+self.target_size, :]
            start = target_chunk[0]
            dx = target_chunk[1][0] - target_chunk[0][0]
            dy = target_chunk[1][1] - target_chunk[0][1]
            theta = math.atan2(dy, dx)
            input_chunk = seq2[is_inside_semicircle(start, theta, 8, seq2)][:self.input_size,:]
            input_chunk, target_chunk = normalize_data(input_chunk, target_chunk, start, theta)
            input_chunk = add_noisy_data(input_chunk)
            samples.append((input_chunk[torch.randperm(input_chunk.size(0))] , target_chunk))
            k+=5

        return samples  # Return list of (input, target) pairs





def is_inside_semicircle(center, direction, radius, cones):
    vectors = cones[:, :2] - center[:2]
    angles = torch.atan2(vectors[:, 1], vectors[:, 0])
    dists = torch.norm(vectors, dim=1)
    
    start_angle = direction - math.pi/4
    end_angle = direction + math.pi/4

    angle_in_range = (angles >= start_angle) & (angles <= end_angle)
    dist_in_range = (dists <= radius) & (dists>1)

    return angle_in_range & dist_in_range

def resample_curve(points, n_points):
    """Resample a 2D curve to have exactly n_points using linear interpolation."""
    # Compute cumulative arc length
    deltas = np.diff(points, axis=0)
    dists = np.sqrt((deltas**2).sum(axis=1))
    cumulative_dist = np.insert(np.cumsum(dists), 0, 0)
    total_dist = cumulative_dist[-1]
    new_distances = np.linspace(0, total_dist, n_points)

    # Interpolate x and y separately
    x_interp = np.interp(new_distances, cumulative_dist, points[:, 0])
    y_interp = np.interp(new_distances, cumulative_dist, points[:, 1])
    return np.stack((x_interp, y_interp), axis=1)

def generate_mid_track(left, right, upsample_factor=5, smoothing=0.5):
    n_points = min(len(left), len(right))

    # Resample both tracks to same length
    left_resampled = resample_curve(left, n_points)
    right_resampled = resample_curve(right, n_points)

    # Midpoints
    midpoints = (left_resampled + right_resampled) / 2

    # Smooth and upsample using spline
    tck, u = splprep([midpoints[:, 0], midpoints[:, 1]], s=smoothing)
    u_fine = np.linspace(0, 1, n_points * upsample_factor)
    x_smooth, y_smooth = splev(u_fine, tck)

    return np.stack((x_smooth, y_smooth), axis=1)


def add_noisy_data(input_chunk):
  for i in range(input_chunk.shape[0]):
    if random.random()>0.9999:
      if input_chunk[i][2] == 1:
        input_chunk[i][2] = 0
      else:
        input_chunk[i][2] = 1

    input_chunk[i][0] += (random.random() - 0.5)
    input_chunk[i][1] += (random.random() - 0.5)

    if random.random()>0.97:
      input_chunk[i][0] = random.random()*10
      input_chunk[i][1] = random.random()*10

    if random.random()>0.97:
      input_chunk[i] = 0

  return input_chunk


def normalize_data(input_chunk, target_chunk, center, theta):
    # Ensure float32 dtype for PyTorch operations
    input_chunk = input_chunk.clone().float()
    target_chunk = target_chunk.clone().float()
    center = center.float()

    # Shift to origin
    noise = torch.tensor([
        np.random.uniform(-2, 2),
        np.random.uniform(0, 2)
    ], dtype=torch.float32)
    
    starting_point = center + noise

    target_chunk = target_chunk - starting_point
    input_chunk[:, :2] = input_chunk[:, :2] - starting_point[:2]

    # Compute angle


    # Rotate to make heading point up (along +y)
    rotation_angle = math.pi/2 - theta + random.uniform(-0.4,0.4)
    cos_a = math.cos(rotation_angle)
    sin_a = math.sin(rotation_angle)

    rotation_matrix = torch.tensor([[cos_a, -sin_a], [sin_a, cos_a]], dtype=torch.float32)

    input_chunk[:, :2] = input_chunk[:, :2] @ rotation_matrix.T
    target_chunk[:, :2] = target_chunk[:, :2] @ rotation_matrix.T

    return input_chunk, target_chunk


def pad_and_mask(src, mask):
    seq_len, feat_dim = src.shape
    masked = src[mask]  # Extract valid elements
    max_len = masked.shape[0] if masked.shape[0] > 0 else 1
    padded = torch.zeros((max_len, feat_dim), device=src.device)
    padded[:masked.shape[0], :] = masked
    return padded


def fast_interleave(left_selected, right_selected):
    L, R = left_selected.shape[0], right_selected.shape[0]
    max_len = max(L, R)
    pad_left = F.pad(left_selected, (0, 0, 0, max_len - L))
    pad_right = F.pad(right_selected, (0, 0, 0, max_len - R))
    return torch.stack((pad_left, pad_right), dim=1).view(-1, 4)


def reverse_track(src, tgt):
    # Reverse both input (src) and target (tgt)
    src_reversed = torch.flip(src, dims=[0]).clone()
    tgt_reversed = torch.flip(tgt, dims=[0])

    # Masks
    left_mask = src_reversed[:, 2] == 1
    right_mask = src_reversed[:, 3] == 1

    # Swap using copies to avoid overlap issues
    new_left = src_reversed[:, 2].clone()
    new_right = src_reversed[:, 3].clone()

    new_left[left_mask] = 0
    new_right[left_mask] = 1

    new_right[right_mask] = 0
    new_left[right_mask] = 1

    src_reversed[:, 2] = new_left
    src_reversed[:, 3] = new_right

    return src_reversed, tgt_reversed



def collate_fn(batch):
    """
    Custom collate function to flatten samples and pad sequences for batching.
    """
    flattened_samples = [sample for sublist in batch for sample in sublist]  # Flatten list of lists
    input_batch, target_batch = zip(*flattened_samples)  # Separate inputs and targets

    # Convert to tensors and pad
    input_padded = pad_sequence(input_batch, batch_first=True, padding_value=0)
    target_padded = pad_sequence(target_batch, batch_first=True, padding_value=0)

    return input_padded, target_padded

def output_data_loader(src_path, tgt_path, src2_path):
    train_dataset = SeqDataset(src_path=src_path, tgt_path=tgt_path,src2_path=src2_path)


    train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, collate_fn=collate_fn)
    return train_dataloader

In [20]:
src_path = "/kaggle/input/deeplearning/data/train/src/" # Replace with the path to your directory
tgt_path = "/kaggle/input/deeplearning/data/train/tgt/"

src2_path = "/kaggle/input/deeplearning/data/train2/src/"

val_src_path = "/kaggle/input/deeplearning/data/val/src/"
val_tgt_path= "/kaggle/input/deeplearning/data/val/tgt/"

val_src2_path = "/kaggle/input/deeplearning/data/val2/src/"

In [21]:
# For training (Noisy)
train_dataloader = output_data_loader(src_path, tgt_path, src2_path) 

In [22]:
# For testing (Clean)
val_dataset = SeqDataset(val_src_path, val_tgt_path, val_src2_path, train=False)
train_dataloader = DataLoader(val_dataset, batch_size=2, collate_fn=collate_fn)

In [23]:
# Move to the root
%cd /kaggle/working/

# Remove all folders (Be careful, this deletes the RacingTeam_DeepLearning folder)
!rm -rf models

/kaggle/working


In [25]:
# 1. Create the directory (the -p flag prevents errors if it already exists)
!mkdir -p models/exported

# 2. Copy the model file from your input directory to the new folder
# !cp "/kaggle/input/models/best_model0.5mse  oldangle   (11).pt" models/exported/best_model.pt

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

def angle_loss(pred, target, eps=1e-6):
    # pred, target: [batch_size, seq_len, 2]
    v_ref = target[:, 1:, :] - target[:, :-1, :]    # Reference segments
    v_pred = pred[:, :-1, :] - target[:, :-1, :]    # Predicted step vectors

    # Dot product and norms
    dot = (v_ref * v_pred).sum(dim=2)
    norm_ref = torch.norm(v_ref, dim=2)
    norm_pred = torch.norm(v_pred, dim=2)

    cos_theta = dot / (norm_ref * norm_pred + eps) # prevent div by zero
    cos_theta = torch.clamp(cos_theta, -1.0 + eps, 1.0 - eps)
    theta = torch.acos(cos_theta)                  # [batch, seq_len-1]

    return theta.mean()                            # scalar angle loss


# BASE_DIR = Path(__file__).resolve().parent.parent
# CHECKPOINT_DIR = BASE_DIR / "models" / "exported"
# CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = "/kaggle/working/models/exported/best_model.pt"
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

# # --- CONFIG LOADING ---
# with open(BASE_DIR / "config" / "data_path.yaml", "r") as f:
#     data_path = yaml.safe_load(f)
# with open(BASE_DIR / "config" / "train_config.yaml", "r") as f:
#     train_config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
seed = 42
torch.manual_seed(seed)

# --- INITIALIZATION ---
model = Seq2Seq().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = torch.nn.MSELoss()
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# --- RESUME TRAINING LOGIC (Initial Epoch Idea) ---
start_epoch = 0
best_loss = float('inf')

# if CHECKPOINT_PATH.exists():
import os

if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['best_loss']
    print(f"Resuming from Epoch {start_epoch}")
else:
    print("No checkpoint found. Training from scratch.")

def train_one_epoch(dataloader):
    model.train()
    total_loss = 0
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs, targets.shape[1])
        
        mse = criterion(outputs, targets)
        angle = angle_loss(outputs, targets)
        loss = 0.5 * mse + angle
        # loss, metrics = polar_geometric_loss(outputs, targets)
        
        loss.backward()
        
        # Gradient Clipping (Prevents LSTM Exploding Gradients)
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        if batch_idx % 10 == 0:
            print(f"  Batch {batch_idx} | Loss: {loss.item():.6f}")
            
    return total_loss / len(dataloader)

# --- MAIN TRAINING LOOP ---
epochs = 100
patience_counter = 0
early_stop_patience = 12

for epoch in range(start_epoch, epochs):
    print(f"\n--- Epoch {epoch}/{epochs} ---")
    avg_loss = train_one_epoch(train_dataloader)
    
    # Update Learning Rate based on loss plateau
    scheduler.step(avg_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch} Average Loss: {avg_loss:.6f} | LR: {current_lr}")

    # --- EARLY STOPPING & SAVING ---
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss': best_loss,
        }, CHECKPOINT_PATH)
        print(f"New Best Model Saved (Loss: {best_loss:.6f})")
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{early_stop_patience}")

    if patience_counter >= early_stop_patience:
        print("Early stopping triggered. Training stopped.")
        break

Using device: cpu
No checkpoint found. Training from scratch.

--- Epoch 0/100 ---
  Batch 0 | Loss: 8.490371
  Batch 10 | Loss: 7.470461
  Batch 20 | Loss: 6.590463
  Batch 30 | Loss: 5.336145
  Batch 40 | Loss: 4.433871
  Batch 50 | Loss: 3.955096
  Batch 60 | Loss: 4.306808
  Batch 70 | Loss: 3.946328
  Batch 80 | Loss: 3.988694
  Batch 90 | Loss: 3.926673
  Batch 100 | Loss: 3.866381
  Batch 110 | Loss: 3.379622
  Batch 120 | Loss: 3.252505
  Batch 130 | Loss: 3.175850
  Batch 140 | Loss: 3.090012
  Batch 150 | Loss: 2.826231
  Batch 160 | Loss: 2.579224
  Batch 170 | Loss: 2.630177
  Batch 180 | Loss: 2.677255
  Batch 190 | Loss: 2.487800
  Batch 200 | Loss: 2.397666
  Batch 210 | Loss: 2.380631
  Batch 220 | Loss: 2.197069
  Batch 230 | Loss: 1.987342
  Batch 240 | Loss: 2.083271
  Batch 250 | Loss: 2.078903
  Batch 260 | Loss: 2.076225
  Batch 270 | Loss: 1.919842
  Batch 280 | Loss: 1.972539
  Batch 290 | Loss: 1.549198
  Batch 300 | Loss: 1.676259
  Batch 310 | Loss: 1.852021


In [ ]:
fig, ax = plt.subplots()
def track(val_dataloader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    checkpoint = torch.load("/kaggle/working/models/exported/best_model.pt", map_location=device)
    model.to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    i = 0
    for batch_idx, (input, target) in enumerate(val_dataloader):



        input, target = input.to(device), target.to(device)


        pred = model(input, target.shape[1]).detach().numpy()  # Forward pass

        outputs = pred

        ax.clear()
        for i in range(input.shape[0]):
            if i >10:
                return

            ax.clear()
            blue_cones = inputs[i][inputs[i][:, 2] == 1]
            yellow_cones = inputs[i][inputs[i][:, 3] == 1]
            
            ax.scatter(blue_cones[:, 0], blue_cones[:, 1], c='blue', s=60, label='Left (Blue)')
            ax.scatter(yellow_cones[:, 0], yellow_cones[:, 1], c='gold', s=60, label='Right (Yellow)')
            
            # Plot Paths
            ax.scatter(targets[i][:, 0], targets[i][:, 1], c='red', s=10, label='Ground Truth')
            ax.scatter(preds[i][:, 0], preds[i][:, 1], c='green', s=10, label='Prediction')
            
            ax.scatter(target[i][:,0], target[i][:,1], c = 'r')
            ax.scatter(outputs[i][:,0], outputs[i][:,1], c = 'g')
            display(fig)
            plt.pause(1)
            i+=1
            
            
track(train_dataloader)